In [1]:
from photutils.detection import DAOStarFinder
from astropy.stats import sigma_clipped_stats
from photutils.aperture import CircularAperture
from astropy.visualization import SqrtStretch
from astropy.visualization.mpl_normalize import ImageNormalize
from astropy.wcs import WCS
from astropy.io import fits
import glob
import os
from scipy.stats import scoreatpercentile

from matplotlib import pyplot as plt
import numpy as np

#import ccdproc

from astropy.nddata import CCDData
import astropy.units as u
from ccdproc import ImageFileCollection, Combiner, combine
from ccdproc import wcs_project

In [2]:
def imdisplay(image, v1perc=10, v2perc=95):
    '''
    display an image 
    OPTIONAL KEYWORD PARAMETERS
    v1perc: one end of the colormap assigned to the v1perc percent lowest flux 
    v2perc: the other end of the colormap assigned to the v2perc percent highest flux    
    '''
    # make sure image is an np array
    nimage = np.array(image)
    # determine the pixel values at the 10th and 95th percentile
    v1 = scoreatpercentile(nimage,v1perc)
    v2 = scoreatpercentile(nimage,v2perc)
    # display using imshow
    #
    # you can play with alternate cmaps in the function below, such as "viridis" or "gray"
    # The 'gray_r' color map reverses the color-scale so that dark display pixels are the brightest in the image
    #
    # vmin and vmax set the min and max pixel values that
    # will be mapped to the extremes of the colormap
    plt.imshow(nimage,cmap='gray_r',vmin=v1,vmax=v2)
    plt.colorbar(fraction=.08)
    
def add_stars(xstars,ystars,radius=20,color='r',ax=None):
    ''' this will add circles at the location of xstars,ystars '''
    if ax is None:
        ax = plt.gca()
    allcircles=[]
    #The zip function takes iterables (e.g. lists or arrays) and groups them into tuples that contain 
    #matched pairs of each input, e.g [(x,y)]
    for x,y in zip(xstars,ystars):
        c = plt.Circle((x,y),radius,color=color,fill=False)
        #allcircles.append(c)
        ax.add_artist(c)

In [4]:
from astropy.io import fits
from photutils.psf import fit_fwhm

def calc_fwhm(imagefilename, xyposlist):
    '''
    This function takes as input an image file name and a list of star pixel coordinates 
    and prints the FWHM for each star.  

    Stars should be bright, unsaturated, and isolated.

    INPUTS: 
    imagefilename: a string containing the image file name

    xyposlist: a list of x and y coordinates of the form [(x1,y1), (x2,y2)]
    This can contain an arbitrary number of stars.

    OUTPUT:
    fwhmlist: a list of FWHM
    '''
    image = fits.getdata(imagefilename)
    
    #use the photutils.psf.fit_fwhm function to fit a 2D PSF to the stars.
    fwhm = fit_fwhm(
        image,
        xypos=xyposlist,
        fwhm=4.0,        # initial guess, pixels
        fit_shape=21
    )
    print(f'The FWHM of the stars in {imagefilename} in pixels are {fwhm}')
    return(fwhm)

In [5]:
# Calculate FWHM for M5 G-band

M5Gstars = [(2310,838),(2546,999),(2320,1047),(2512,1177),(1906,1277)]

M5G_fwhm = calc_fwhm('M5 G-band image.fits', M5Gstars)

print("G-band FWHM:", M5G_fwhm)
print("Mean G-band FWHM:", np.mean(M5G_fwhm))
print("Median G-band FWHM:", np.median(M5G_fwhm))

The FWHM of the stars in M5 G-band image.fits in pixels are [5.64782557 5.60401002 5.8534596  5.86903849 6.09410038]
G-band FWHM: [5.64782557 5.60401002 5.8534596  5.86903849 6.09410038]
Mean G-band FWHM: 5.813686811931993
Median G-band FWHM: 5.853459595558811


In [6]:
# Calculate FWHM for M5 R-band

M5Rstars = [(2322,1044),(2545,1001),(2673,928),(2776,1075),(2652,1309)]

M5R_fwhm = calc_fwhm('M5 R-band image.fits', M5Rstars)

print("R-band FWHM:", M5G_fwhm)
print("Mean R-band FWHM:", np.mean(M5R_fwhm))
print("Median R-band FWHM:", np.median(M5R_fwhm))

The FWHM of the stars in M5 R-band image.fits in pixels are [6.3381363  6.34301261 6.41543124 6.33026496 6.93942727]
R-band FWHM: [5.64782557 5.60401002 5.8534596  5.86903849 6.09410038]
Mean R-band FWHM: 6.473254476751111
Median R-band FWHM: 6.343012609597192
